# First Agentic Framework project!!  
We're going to build a simple Agent system for generating cold sales outreach emails:  
1. Agent Workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

### Importatnt  
Instructor used `SendGrid` as email provider, it's optional to select and use the email provider of your choice.  

**Instructions for Setting up `SendGrid`**  

Visit : https://sendgrid.com/  

(SendGrid is a Twilio company for sending emails.)   

Setting up a SendGrid account is free! (at least, now)  
Once you've created an account, click on:  

Settings (left sidebar or look for settings) >> API keys >> Create API Key (button)  

Copy the key then add a new line to your .env file:  
`SENDGRID_API_KEY=xxxxxxx`  

And also, within SendGrid, go to:  

Settings >> Sender Authentication >> "Verify a single sender" and verify that your own email address is a real email address, so that SendGrid can send emails for you.

In [3]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [4]:
load_dotenv(override = True)

True

In [5]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("coast092@gmail.com")  # Change to your verified sender
    to_email = To("coast092@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()


202


### Received the email?  
In previous cell output, if you got response code `202`, all is set, check inbox/spam/junk.

#### Certfificate error  
If you get an error SSL: CETRIFICATE_VERIFY_FAILED, then  
First run this: `!uv pip install --upgrade certifi`  
Next run this:  
`import certifi`  
`import os`  
`os.environ['SSL_CERT_FILE'] = certifi.where()`

### Other errors or no-email  
If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard.  


## Step 1: Agent Workflow

In [6]:
instructions1 = "You are a sales agent working for CoplAI, \
    a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
    You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
    a company that provides a Saas tool for ensuring SOC2 compliance and preparing for audits, powered by AI, \
    You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
    a company that provides as SaaS tool for ensuing SOC2 compliance and preparing for audits, powered by AI. \
    You write concise, to the point col emails."

In [9]:
sales_agent1 = Agent(
    name = "Professional Sales Agent",
    instructions = instructions1,
    model = 'gpt-4o-mini'
)

sales_agent2 = Agent(
    name = "Engaging Sales Agent",
    instructions = instructions2,
    model = 'gpt-4o-mini'
)

sales_agent3 = Agent(
    name = "Busy Sales Agent",
    instructions = instructions3,
    model = 'gpt-4o-mini'
)

In [10]:
result = Runner.run_streamed(sales_agent1, input = 'Write a cold sales email')
async for event in result.stream_events():
    if event.type == 'raw_response_event' and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = '', flush= True)

Subject: Streamline Your SOC 2 Compliance with AI-Powered Solutions

Hi [Recipient’s Name],

I hope this message finds you well. My name is [Your Name], and I represent CoplAI, where we specialize in simplifying SOC 2 compliance through our advanced SaaS tool, powered by AI.

In today's rapidly evolving regulatory landscape, maintaining compliance can be a daunting task. Our platform automates much of the process, helping organizations like yours not only adhere to SOC 2 standards but also enhance their operational efficiency.

Key benefits of our solution include:

- **Automated Documentation**: Reduce the tedious manual work involved in preparing for audits.
- **Real-Time Monitoring**: Stay informed with continuous compliance status updates.
- **Risk Assessment**: Proactively identify and mitigate potential compliance risks.

We’ve helped companies similar to yours save significant time and resources during their audit processes. I would love to connect for a brief conversation to di

In [11]:
message = "write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")

Subject: Simplify Your SOC 2 Compliance Journey with AI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am with CoplAI, where we specialize in streamlining the SOC 2 compliance process for organizations like yours.

Navigating SOC 2 compliance can be daunting, requiring significant resources and time. Our AI-driven SaaS tool is designed to simplify this journey by automating key compliance tasks, ensuring that you are audit-ready without the usual headaches.

Here are a few ways our solution can benefit your organization:

- **Efficiency:** Reduce compliance preparation time by up to 50%, allowing your team to focus on core business activities.
- **Accuracy:** Leverage AI to minimize human error and ensure all necessary documentation is in place.
- **Scalability:** Easily adapt to changing compliance requirements as your organization grows.

I would love the opportunity to discuss how CoplAI can support your compliance goals and help you ach

In [13]:
sales_picker = Agent(
    name = "Sales Picker",
    instructions = "You pick the best cold sales email from the given options. \
        Imagine you are a customer and pick the one you are most likely to respons to. \ \
        Do not give an explanation; reply with the selected email only.",
    
    model = 'gpt-4o-mini'
)

<>:4: SyntaxWarning: invalid escape sequence '\ '
<>:4: SyntaxWarning: invalid escape sequence '\ '
C:\Users\Paco_Minha\AppData\Local\Temp\ipykernel_17196\1330623796.py:4: SyntaxWarning: invalid escape sequence '\ '
  Imagine you are a customer and pick the one you are most likely to respons to. \ \


In [14]:
message = 'Write a cold sales email'

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    outputs = [result.final_output for result in results]
    emails = "Cold Sales Emails: \n\n" + "\n\nEmail: \n\n".join(outputs)
    best = await Runner.run(sales_picker, emails)
    print(f"Best Sales Email: \n{best.final_output}")

Best Sales Email: 
Subject: Ready to Tame Your SOC2 Compliance Chaos? 🦁

Hey [Recipient's Name],

I hope this email finds you well, or at least more relaxed than an auditor on a beach vacation (which, as we know, is rare)!

I’m reaching out because I imagine you’re probably juggling a million things at once—keeping the business running, ensuring security, and dodging audits like a game of dodgeball. What if I told you there’s a way to simplify your SOC2 compliance and make audits feel more like a walk in the park than a trek through a minefield?

Meet ComplAI: your new best friend in the world of compliance. With our AI-driven SaaS tool, you can streamline your SOC2 processes, automate evidence collection, and even gain insights faster than you can say “compliance nightmare.” 

(Plus, we promise it’s way less painful than trying to decipher the fine print of the latest compliance guide!)

Imagine spending less time flipping through binders and more time strategizing your next big proje

Now go and check out the trace:

https://platform.openai.com/traces

## Step 2: Use of Tools   
Add A tool to the mix.  
Remember all that json boilerplate and `hanlde_tool_calls()` function with the if logic....

In [15]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [16]:
sales_agent1


Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for CoplAI,     a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI.     You write professional, serious cold emails.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None, retry=None, context_management=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 02 and 03: Tools and Agent Interactions  
Remembere all that json boilerplate?  
Simply wrap the json with the decorator @function_tool

In [17]:
@function_tool
def send_email(body: str):
    """
    Send out an email ith the given body to all sales prospects
    """
    sg = sendgrid.SendGridAPIClient(api_key = os.environ.get('SENDGRID_API_KEY'))
    from_email = Email('coast092@gmail.com')
    to_email = To('coast092@gmail.com')
    content = Content("text/plain", body)
    mail = Mail(from_email, to_email, "Sales email", content).get()
    sg.client.main.send.post(request_body=mail)
    return {"status" : "success"}

#### This has Automatically been converted into a tool, with the boilerplate json created.

In [20]:
# let's have a look
send_email

FunctionTool(name='send_email', description='Send out an email ith the given body to all sales prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000221B7055F10>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [21]:
tool1 = sales_agent1.as_tool(tool_name = 'sales_agent1', 
                             tool_description='write a cold sales email')

tool1

FunctionTool(name='sales_agent1', description='write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000221B7B3A1B0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

#### So now we can gather all the tools togather  
A tool for each of our 3 email-writing agetns  
And a tool for our function to send emails

In [22]:
description = 'Write a col sales email'

tool1 = sales_agent1.as_tool(tool_name = 'sales_agent1', tool_description=description)
tool2 = sales_agent2.as_tool(tool_name = 'sales_agent2', tool_description = description)
tool3 = sales_agent3.as_tool(tool_name = 'sales_agent3', tool_description=description)

tools = [tool1, tool2, tool3, send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a col sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000221B7B39E50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='sales_agent2', description='Write a col sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, o

### And now it's time for our Sales Manager - our planning agent

In [34]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales agent tools.

Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate threee different email drafts. Do not proceed until all three drafts are ready.
2. Evaluate and Select: Review the drafts and choose the single best email usig your judgment of which one is most effective.
3. Use the send_email tool to send the best email (and only the best email) to the user.

Crucial Rules:
- You must use the sales agent tools to generate the drafts - do not write them yourself.
- You must send ONE email using the send_email tool - never more than one.

"""

sales_manager = Agent(
    name = "Sales Manager",
    instructions = instructions, 
    model = 'gpt-4o-mini',
    tools = tools,
)

message = "Send a cold sales email addressed to 'Dear CEO' "
with trace("Sales Manager"):
    result = await Runner.run(sales_manager, message)
    print(result.final_output)


The final email has been prepared but not sent directly. Here's a summary of the cold sales email that will be sent out:

---

**Subject:** Streamline Your SOC 2 Compliance Process with ComplAI

Dear CEO,

I hope this message finds you well.

As the landscape of data security continues to evolve, ensuring SOC 2 compliance can be a daunting task for many organizations. At ComplAI, we leverage advanced AI technology to simplify and automate the compliance process, enabling your team to focus on what truly matters: growing your business.

With our platform, you can expect to reduce the time and resources spent on preparing for audits while increasing the accuracy and reliability of your compliance efforts. This not only minimizes operational costs but also enhances your organization's reputation and trustworthiness.

I would appreciate the opportunity to discuss how ComplAI can address your specific compliance challenges. Would you be available for a brief call next week?

Thank you for c

### Handoffs represent a way an agent can delegate to an agent, passing cotnrol to it   
Handoffs and Agents-as-tools are similar:  
In both cases, an Agent can collaborate with another Agent  
With tools, control passes back  
With handoffs, control passess across.


In [27]:
subject_instructions = "You can write a subject for a cold sales email. \
    You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
    You are given a text email body which might have some markdown \
        and you need to convert it to an HTML email body with simple, lear, compelling layout and design."

subject_writer = Agent(name = "Email subject writer",
                       instructions = subject_instructions,
                       model = 'gpt-4o-mini')

subject_tool = subject_writer.as_tool(tool_name="subject_writer",
                                      tool_description = "Write a subject for a cold sales email" )

html_converter = Agent(name = "HTML email body converter",
                       instructions = html_instructions,
                       model = 'gpt-4o-mini')

html_tool = html_converter.as_tool(tool_name = 'html_converter',
                                   tool_description='Convert a text email body to an HTML email body')

In [28]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """Send out an email with the given subject and HTML body to all sales prospects"""
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get("SENDGRID_API_KEY"))
    from_email = Email("coast092@gmail.com") 
    to_email = To("coast092@gmail.com")
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return  {"status" : 'success'}


In [29]:
tools = [subject_tool, html_tool, send_html_email]
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000221B7B2FB90>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 

In [30]:
instructions = "You are an email formatter and sender. You receive the body of an email to be send. \
    You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
    Finally, you use the send_html_email tool to send the email with the subject and HTML body."

email_agent = Agent(
    name = "Email Manager",
    instructions = instructions, 
    tools = tools, 
    model = 'gpt-4o-mini',
    handoff_description= "Convert an email to HTML and send it"
)

### Now we have 3 tools and 1 handoff

In [31]:
tools = [tool1, tool2, tool3]
handoffs = [email_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a col sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000221B7B39E50>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None), FunctionTool(name='sales_agent2', description='Write a col sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on

In [33]:
sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
    name = "Sales Manager",
    instructions = sales_manager_instructions,
    tools= tools,
    handoffs = handoffs,
    model = 'gpt-4o-mini',

)

message = 'Send out a cold sales email addressed to Dear CEO from XYZ'

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)
    print(result.final_output)

The cold sales email has been successfully sent! If you need any further assistance or want to draft another email, feel free to let me know.


### Remember to check the trace and then your inbox/spam/junk  
https://platform.openai.com/traces